In [21]:
from torch import nn
import numpy as np
import matplotlib as mlp
import pandas as pd
import h5py
from time import sleep
from torch.utils.data import Dataset, DataLoader, random_split, Subset

In [22]:
class star_tracker_v1(nn.Module):
    def __init__(self, n_bins, n_classes):
        super().__init__()

        self.bn1 = nn.BatchNorm1d(n_bins) 
        
        self.grp1 = nn.Sequential(
            nn.Linear(n_bins, 100),
            nn.ReLU(),
            nn.BatchNorm1d(100),
            nn.Dropout(p=0.3),
        )

        self.grp2 = nn.Sequential(
            nn.Linear(250, n_classes),
            nn.BatchNorm1d(n_classes),
            nn.Dropout(p=0.3),
        )
        
    def forward(self, x): 
        x = self.bn1(x)
        x = self.grp1(x)
        x = self.grp2(x)
        return x    

In [23]:
class H5Data(Dataset):
    def __init__(self, path):
        self.path = path
        self.index_map = []
        self.file = h5py.File(path, "r")
        for group in self.file.keys():
            for sample in self.file[str(group)]:
                self.index_map.append((group, sample))
                    
        self.length = len(self.index_map)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        group, sample = self.index_map[idx]
        data = self.file[group][sample][:]

        return torch.tensor(data, dtype=torch.float32), int(float(group))

In [24]:
path = "./data.hdf5"
data = H5Data(path)

In [25]:
from sklearn.model_selection import train_test_split

labels = [int(float(group)) for group, _ in data.index_map]
seed = 1
train_idx, test_idx = train_test_split(
    range(len(data.index_map)),
    train_size=0.8,
    stratify=labels,
    random_state=seed
)
train_sub = Subset(data, train_idx)
test_sub = Subset(data, test_idx)

BATCH_SIZE = 32
train_dataloader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test_sub, batch_size=BATCH_SIZE)
print(f"Length of train dataloader: {len(train_dataloader)}")
print(f"Length of test dataloader: {len(test_dataloader)}")

Length of train dataloader: 22573
Length of test dataloader: 5644


In [26]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = star_tracker_v1(20, 9029)
model.to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params=model.parameters(), lr=0.01)

In [27]:
from tqdm import tqdm
epochs = 100
for epoch in tqdm(range(epochs)):
    print(f"Epoch: {epoch}")

    train_loss, train_accuracy = 0, 0
    model.train()
    for batch, (x, y) in enumerate(train_dataloader):
        x, y = x.to(device), y.to(device)

        y_pred = model(x)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        train_pred = torch.argmax(y_pred, dim=1)
        train_accuracy += (train_pred == y).sum().item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss /= len(train_dataloader)
    train_accuracy /= len(train_dataloader) * BATCH_SIZE

    test_loss, test_accuracy = 0, 0
    model.eval()
    with torch.inference_mode():
        for x,y in (test_dataloader):
            x, y = x.to(device), y.to(device)

            y_pred = model(x)
            test_loss += loss_fn(y_pred, y).item()
            test_pred = torch.argmax(y_pred, dim=1)
            test_accuracy += (test_pred == y).sum().item()

        test_loss /= len(test_dataloader)
        test_accuracy /= len(test_dataloader) * BATCH_SIZE

    print(f"\nTrain loss: {train_loss:.5f} | Train Accuracy: {train_accuracy:.5f} | Test loss: {test_loss:.5f}, Test acc: {test_accuracy:.2f}%\n")

  0%|                                                   | 0/100 [00:00<?, ?it/s]

Epoch: 0


  1%|▍                                       | 1/100 [04:25<7:18:40, 265.86s/it]


Train loss: 9.21416 | Train Accuracy: 0.00013 | Test loss: 3148.17752, Test acc: 0.00%

Epoch: 1


  2%|▊                                       | 2/100 [07:53<6:18:02, 231.45s/it]

...


Train loss: 8.71137 | Train Accuracy: 0.00054 | Test loss: 258.29071, Test acc: 0.00%

Epoch: 25


 26%|█████████▌                           | 26/100 [2:00:59<4:45:13, 231.27s/it]


Train loss: 8.70674 | Train Accuracy: 0.00055 | Test loss: 238.46998, Test acc: 0.00%

Epoch: 26


 26%|█████████▌                           | 26/100 [2:01:53<5:46:54, 281.28s/it]


KeyboardInterrupt: 

In [28]:
labels = [int(float(group)) for group, _ in data.index_map]
print("Max label:", max(labels))
print("Min label:", min(labels))
print("Num unique labels:", len(set(labels)))

Max label: 9110
Min label: 1
Num unique labels: 9029
